<a href="https://colab.research.google.com/github/appling2024/MSP/blob/Liza/Summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive #подключаем гугл диск
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Задание:  автоматически создать аннотации (краткие описания) для статей, у которых они отсутствуют (2002, 2004, 2005, 2006, 2008, 2011 годы). План:
- прочитать статьи из корпуса;
- сгенерировать аннотации двумя способами: с помощью алгоритма **LexRank** и нейросети **RuT5**;
- сохранить полученные аннотации в отдельные папки по годам.

In [5]:
import os

In [6]:
years = [2002, 2004, 2005, 2006, 2008, 2011] #статьи по годам, у которых нет аннотаций в корпусе

In [7]:
path = f'/content/drive/MyDrive/full_corpus/'

In [8]:
texts = [] #список для хранения наших данных

In [9]:
for root, dirs, files in os.walk(path): #итерация по файлам
  for file in files:
      if 'annotation' not in file: #отбираем только те файлы, где нет 'annotation'
        with open(os.path.join(root, file), 'r') as f: #читаем содержимое
              text = f.read()
              name = str(file[:-4]) #имя файла
              year = str(file[-8:-4]) #год из имени файла
              texts += [(name, text, year)] #добавляем кортеж в список

In [10]:
texts[:1]

[('Martynenko_CL_thesis_2004_lem',
  'г.я. мартыненко (спбгу, санкт-петербург) корпус в контекст общий теория ценоз 1. в последний год постоянно расти интерес к изучение сообщество (ценозов) сам разнообразный природы: в биология исследуться сообщество организм (биоценозы), в документалистика - массив научный публикаций, в науковедение - научный коллективы, в технетик - ансамбль изделий, машин, механизмов, в экономика - объединение предприятий, в политология - группировка государств, в языкознание - совокупность лексический единиц, в спорт - множество команд, образовать лиги, дивизионы, конференция и т.п. 2. при исследование перечисленный и многий другой сообщество можно найти много общий как в сущностном, так и в методический отношении. исторически пальма первенство в исследование сообщество принадлежать статистике, а в последний время статистический идея всё более обогащаться теоретико-классификационный системными, социально-психологический и синергетический представлениями. в настоящ

In [11]:
import pandas as pd

In [12]:
df = pd.DataFrame(texts, columns=['names', 'texts', 'years'])
#создаем таблицу полученного списка
df

,names,texts,years
0,Martynenko_CL_thesis_2004_lem,"г.я. мартыненко (спбгу, санкт-петербург) корпу...",_lem
1,Godgildieva_IMS_2016_lem,создание словарь валентность русский язык на о...,_lem
2,Dao_CL_2008_lem,дао хонг тху семантический подбор параллельный...,_lem
3,Masevich_IMS_2018_lem,вариативность представление имя политический д...,_lem
4,Shevchenko_CL_2005_lem,"и.в. шевченко, а.г. рабулец, в.а. широков (укр...",_lem
...,...,...,...
658,Sayama_CL_2021_lem,г. сая влияние частотность на форма падежный о...,_lem
659,Kobozeva_CL_2019_lem,"м.в. кобозева, д.б. писаревская, а.а. тугутова...",_lem
660,Dikaryova_CL_2005_lem,"с.с. дикарева, ю.в .пигарев, н.а. голубец (тав...",_lem
661,Kretov_CL_thesis_2004_lem,а.а. крет (воронежский государственный универс...,_lem


Sumy — это инструмент для автоматического обобщения текстов на Python. В основе работы лежит задача резюмирования текста, которая позволяет получать краткие и четкие версии больших объемов информации.

Преимущества:
- Sumy поддерживает несколько методов обобщения, включая LSA, TextRank и LexRank.
- С минимальным количеством кода можно начать обобщение текста довольно быстро.
- Sumy легко интегрируется с другими Python-библиотеками.
- Помимо английского, Sumy также предоставляет поддержку для русского языка.


In [13]:
pip install sumy

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.3/97.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 77.5 MB/s eta 0:00:00
  Created wheel for breadability: filename=breadability-0.1.20-py2.py3-none-any.whl size=21693 sha256=b157da14939b8d6c06706c521c07f2aa4c24b1c446e44088deff7f905e468112
  Stored in directory: /root/.cache/pip/wheels/4d/57/58/7e3d7fedf51fe248b7fcee3df6945ae28638e22cddf01eb92b
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=d84a182aa371b98997359144746db428547daa3f77e8e237aef225c495adbeda
  Stored in directory: /root/.cache/pip/wheels/1a/b0/8c/4b75c4116c31f83c8f9f047231251e13cc74481cca4a78a9ce
Successfully built breadability docopt


In [14]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer #импортируем необходимые модули

!pip install nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') #загружаем токенизатор для РЯ

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [15]:
summarizer = LexRankSummarizer() #суммаризатор LexRank

In [16]:
abstracts = [] #список для хранения аннотаций

In [17]:
for text in df.texts: #итерация
  doc = text
  parser = PlaintextParser.from_string(doc, Tokenizer('russian'))
  summary = summarizer(parser.document, 3) #генерируем аннотации для каждого текста по 3 предложения
  abstracts.append([str(sentence) for sentence in summary])

In [18]:
abstracts[:10]

[['с точка зрение формальный логика ценоз относиться к класс собирательный понятий, в который отразить признак группа единиц, образовать единый целое.',
  'в теория статистика ценоз рассматриваться как естественный совокупность (а.а. чупров), который представлять себя множество объектов, локализовать в определённый рамка время и пространство и образовать единый целое.',
  'для лингвист характерный суммативный подход - стремление включить в корпус максимальный число текстов, для он представительность - это в первый очередь размер корпуса.'],
 ['у семантический роль с больший частота обычно слишком общий значение, чтобы они можно быть использовать такой образом.',
  'для примера, подлежащее при глагол родить в прямой значение мочь быть только женщина, поэтому роль первый порядок ag логично сузить до роль второй порядок woman:1. для каждый рамка валентность приводиться несколько пример употребления, возможный синонимы, возвратность глагол и значение, в который в она употребляться глагол в

In [19]:
annotations = [] #формируем текст аннотаций из списка предолжений. для этого создаем список

In [20]:
for abstr in enumerate(abstracts, start=1): #начинаем с 1, а не с 0
  print(f'Аннотация статьи {abstr[0]}:') #выводим
  annotation = str(' '.join(abstr[1]))
  annotations.append(annotation)
  print(annotation, '\n\n')

Аннотация статьи 1:
с точка зрение формальный логика ценоз относиться к класс собирательный понятий, в который отразить признак группа единиц, образовать единый целое. в теория статистика ценоз рассматриваться как естественный совокупность (а.а. чупров), который представлять себя множество объектов, локализовать в определённый рамка время и пространство и образовать единый целое. для лингвист характерный суммативный подход - стремление включить в корпус максимальный число текстов, для он представительность - это в первый очередь размер корпуса. 


Аннотация статьи 2:
у семантический роль с больший частота обычно слишком общий значение, чтобы они можно быть использовать такой образом. для примера, подлежащее при глагол родить в прямой значение мочь быть только женщина, поэтому роль первый порядок ag логично сузить до роль второй порядок woman:1. для каждый рамка валентность приводиться несколько пример употребления, возможный синонимы, возвратность глагол и значение, в который в она упо

In [21]:
annots = pd.DataFrame(annotations, columns=['annotations_sumy']) #создаем df с аннотациями sumy и объединяем с исходными данными

In [22]:
sumy = pd.concat([df, annots], join='outer', axis=1) #pandas.concat объединяет два df (df и annots).
#'outer' -- способ объединения по всем индексам, которые есть хотя бы в одном из df
#axis = 1 производит объединение по столбцам, то есть df ставятся рядом (axis = 0 по строкам, то есть друг под другом)

In [23]:
sumy

,names,texts,years,annotations_sumy
0,Martynenko_CL_thesis_2004_lem,"г.я. мартыненко (спбгу, санкт-петербург) корпу...",_lem,с точка зрение формальный логика ценоз относит...
1,Godgildieva_IMS_2016_lem,создание словарь валентность русский язык на о...,_lem,у семантический роль с больший частота обычно ...
2,Dao_CL_2008_lem,дао хонг тху семантический подбор параллельный...,_lem,задача наш работа быть изучение характерный ос...
3,Masevich_IMS_2018_lem,вариативность представление имя политический д...,_lem,на рис. 16 показать кривая частотный поведение...
4,Shevchenko_CL_2005_lem,"и.в. шевченко, а.г. рабулец, в.а. широков (укр...",_lem,множество лексема каждый грамматический класс ...
...,...,...,...,...
658,Sayama_CL_2021_lem,г. сая влияние частотность на форма падежный о...,_lem,для получение объективный и количественный док...
659,Kobozeva_CL_2019_lem,"м.в. кобозева, д.б. писаревская, а.а. тугутова...",_lem,в первый часть инструкция представить определе...
660,Dikaryova_CL_2005_lem,"с.с. дикарева, ю.в .пигарев, н.а. голубец (тав...",_lem,"с.с. дикарева, ю.в .пигарев, н.а. голубец (тав..."
661,Kretov_CL_thesis_2004_lem,а.а. крет (воронежский государственный универс...,_lem,"корпус включать в себя, с один стороны, оригин..."


Transformers — это архитектура нейросетей, которая стала основой большинства современных моделей в области обработки естественного языка (NLP), таких как GPT, BERT, T5, и др.

Архитектура Transformer была предложена в статье "Attention is All You Need" (2017) исследователями из Google. Главная новизна заключалась в использовании механизма внимания (attention) без рекуррентных или сверточных слоев.

Модель обращает внимание на все слова во входной последовательности, чтобы понять контекст.
Поскольку трансформеры не обрабатывают данные последовательно (в отличие от RNN), им нужно явно указывать порядок слов. Это делается с помощью векторного кодирования позиций слов.

Encoder считывает входную последовательность и формирует внутреннее представление.
Decoder на основе этого представления генерирует выход, например, перевод или ответ.

In [24]:
!pip install transformers

In [25]:
from transformers import AutoModelForSeq2SeqLM, T5TokenizerFast

In [26]:
import torch
from transformers import AutoTokenizer, AutoModelWithLMHead, T5ForConditionalGeneration

In [27]:
model_name = "IlyaGusev/rut5_base_sum_gazeta" #загружаем предобученную русскую модель RuT5 для суммаризации
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/828k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/977M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/977M [00:00<?, ?B/s]

In [28]:
for text in df.texts[:5]: #проходимся по первым 5 текстам с df texts
  article_text = text #сохраняем текущий текст в переменную
  input_ids = tokenizer([article_text], max_length=600, add_special_tokens=True, padding="max_length", truncation=True, return_tensors="pt")["input_ids"] #векторизуем тексты, возвращаем все в виду pytorch тензора
  output_ids = model.generate(input_ids=input_ids, no_repeat_ngram_size=4)[0] #генерируем выходной текст
  summary = tokenizer.decode(output_ids, skip_special_tokens=True) #преобразуем токены обратно в строку
  print(f'Аннотация текста:\n {summary}\n\n')

Аннотация текста:
 Корпус в контекст общий теория ценозов можно отнести к классу множества объектов, который представляет себя множество объектов, локализовать время и пространство и образовать единый целое.


Аннотация текста:
 В российском языке создается компьютерный словарь валентность чешский язык verbalex, который позволяет отразить синтаксический и семантический валентность лексики. Для этого необходимо разработать способ автоматического обработки языковой материала для создания словаря.


Аннотация текста:
 Для создания вьетнамско-английский корпуса, в котором мочь действовать вьетнамский язык в соотношении с другой языками, необходимо подготовить текст, который должен быть соответствовать конструкция предложение перевода на английский язык. Это позволит получить первоначальный теоретический вывод о семантике параллельного конструкция в корпусе.


Аннотация текста:
 Введение настоящей публикации посвящено диахроническому исследованию частотный поведение политической лексики в т

In [30]:
for year in years: #Создали папку для каждого года на диске
  os.mkdir(f'/content/drive/MyDrive/corpus_summ{year}')

In [ ]:
annotations_t5 = []
#генерируем аннотации RuT5 для всех текстов и сохраняем обратно с список
for text in df.texts:
  article_text = text
  input_ids = tokenizer([article_text], max_length=600, add_special_tokens=True, padding="max_length", truncation=True, return_tensors="pt")["input_ids"]
  output_ids = model.generate(input_ids=input_ids, min_length=50, max_length=250, no_repeat_ngram_size=4)[0]
  summary = tokenizer.decode(output_ids, skip_special_tokens=True)
  print(f'Аннотация текста:\n {summary}\n\n')
  annotations_t5.append(str(''.join(summary)))

Аннотация текста:
 Корпус в контекст общий теория ценозов можно отнести к классу множества объектов, который представляет себя множество объектов, локализовать время и пространство и образовать единый целое.


Аннотация текста:
 В российском языке создается компьютерный словарь валентность чешский язык verbalex, который позволяет отразить синтаксический и семантический валентность лексики. Для этого необходимо разработать способ автоматического обработки языковой материала для создания словаря.


Аннотация текста:
 Для создания вьетнамско-английский корпуса, в котором мочь действовать вьетнамский язык в соотношении с другой языками, необходимо подготовить текст, который должен быть соответствовать конструкция предложение перевода на английский язык. Это позволит получить первоначальный теоретический вывод о семантике параллельного конструкция в корпусе.


Аннотация текста:
 Введение настоящей публикации посвящено диахроническому исследованию частотный поведение политической лексики в т

In [ ]:
annots_t5 = pd.DataFrame(annotations_t5, columns=['annotations_t5']) #df с аннотациями RuT5

In [ ]:
full = pd.concat([sumy, annots_t5], join='outer', axis=1) #сбор всех результатов

In [ ]:
full